# SOH Forecasting — NASA Cell #23, **corrected** CC/CV formula (with voltage-range correction)

Same pipeline as `soh_forecast_dqdv_nasa_smoothed.ipynb` (1-min split via `Original_Segment`, CC/CV split at `V_CV = 4.199 V`, 75/25 chronological split, Transformer encoder, 5-segment rolling-median smoothing). Only the SOH formula is changed.

**Reference values** come from the first 10 segments:

$$Q_{cc,\mathrm{avg}} = \overline{Q_{cc}}_{[0..9]}, \quad Q_{cv,\mathrm{avg}} = \overline{Q_{cv}}_{[0..9]}, \quad \Delta V_{\mathrm{avg}} = \overline{\Delta V_{cc}}_{[0..9]}$$

**Voltage-range correction** lifts the CC contribution of narrow segments and damps wide ones:

$$K_{cc} = \Delta V_{\mathrm{avg}} / \Delta V_{cc}, \quad Q_{cc,\mathrm{corr}} = K_{cc}^{\alpha}$$

**Corrected SOH components and total:**

$$\mathrm{SOH}_{cc,\mathrm{corr}} = Q_{cc,\mathrm{corr}} \cdot (Q_{cc} / Q_{cc,\mathrm{avg}})$$
$$\mathrm{SOH}_{cv,\mathrm{corr}} = Q_{cv} / Q_{cv,\mathrm{avg}}$$
$$\mathrm{SOH}(\%) = (w_{cc} \cdot \mathrm{SOH}_{cc,\mathrm{corr}} + w_{cv} \cdot \mathrm{SOH}_{cv,\mathrm{corr}}) \times 100$$

Defaults: `alpha = 0.8`, `w_cc = 0.6`, `w_cv = 0.4`.

> Requires: `pandas numpy matplotlib scikit-learn torch`

> **Note on the `_pred` block** from the formula spec: that block reconstructs SOH from predicted `Q_cc` / `Q_cv`. That would require changing the model to forecast `q_cc` and `q_cv` as joint targets (a different setup) — not done here. This notebook keeps SOH as the single forecast target.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

torch.manual_seed(0)
np.random.seed(0)

CSV_PATH = "Cell_23_NASA.csv"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Load and inspect

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
# the first column is an unnamed index; drop or rename
if df.columns[0].startswith("Unnamed"):
    df = df.drop(columns=df.columns[0])
print("rows:", len(df), " columns:", list(df.columns))
print("TYPE values:", df["TYPE"].unique())
print("segments:", df["Original_Segment"].nunique(),
      "  range:", df["Original_Segment"].min(), "..", df["Original_Segment"].max())
df.head()


## 2. Per-segment aggregation with CC/CV split

We group by `Original_Segment` directly. Within each segment, samples below `V_CV = 4.199 V` are CC (voltage rising at constant current) and the remainder is CV (voltage held at ~4.2 V, current tapering). We compute `q_cc`, `dv_cc`, and `q_cv` per segment so the SOH formula can use them. A segment is dropped only if it has fewer than `MIN_ROWS` samples.


In [ ]:
MIN_ROWS = 10     # drop segments shorter than this many samples
V_CV     = 4.199  # V: threshold separating CC (below) from CV (above)

def aggregate_segment(g):
    if len(g) < MIN_ROWS:
        return None
    t_start = g["TIME"].iloc[0]
    t_end   = g["TIME"].iloc[-1]
    voltage = g["VOLTAGE"].values
    q_vals  = g["Q"].values

    # CC/CV split: first sample where V >= V_CV is the boundary
    above = np.where(voltage >= V_CV)[0]
    if len(above) > 0:
        trans = above[0]
        if trans > 0:
            q_cc  = float(q_vals[trans-1] - q_vals[0])
            dv_cc = float(voltage[trans-1] - voltage[0])
            q_cv  = float(q_vals[-1]      - q_vals[trans-1])
        else:
            q_cc, dv_cc = 0.0, 0.0
            q_cv = float(q_vals[-1] - q_vals[0])
    else:
        q_cc  = float(q_vals[-1]  - q_vals[0])
        dv_cc = float(voltage[-1] - voltage[0])
        q_cv  = 0.0

    return pd.Series({
        "t_start":        t_start,
        "t_end":          t_end,
        "duration_min":   (t_end - t_start) / 60.0,
        "v_start":        float(voltage[0]),
        "v_end":          float(voltage[-1]),
        "delta_v":        float(voltage[-1] - voltage[0]),
        "v_mean":         g["VOLTAGE"].mean(),
        "v_max":          g["VOLTAGE"].max(),
        "v_min":          g["VOLTAGE"].min(),
        "i_mean":         g["CURRENT"].mean(),
        "i_max":          g["CURRENT"].max(),
        "i_min":          g["CURRENT"].min(),
        "temp_mean":      g["TEMPERATURE"].mean(),
        "temp_max":       g["TEMPERATURE"].max(),
        "temp_min":       g["TEMPERATURE"].min(),
        "temp_spread":    g["TEMPERATURE"].max() - g["TEMPERATURE"].min(),
        "cap_charged_ah": float(q_vals[-1] - q_vals[0]),
        "energy_wh":      g["Energy_Wh"].iloc[-1] - g["Energy_Wh"].iloc[0],
        "n_samples":      len(g),
        "q_cc":           q_cc,
        "dv_cc":          dv_cc,
        "q_cv":           q_cv,
    })

records = []
for sid, g in df.groupby("Original_Segment", sort=True):
    rec = aggregate_segment(g)
    if rec is not None:
        rec["segment_id"] = sid
        records.append(rec)

cycles = pd.DataFrame(records).reset_index(drop=True)
cycles["cycle_idx"] = np.arange(len(cycles))
cycles["days_since_start"] = (cycles["t_start"] - cycles["t_start"].iloc[0]) / 86400.0
print(f"Kept {len(cycles)} segments out of {df['Original_Segment'].nunique()} (after MIN_ROWS filter)")
print("per-segment q_cc / dv_cc / q_cv distributions:")
print(cycles[['q_cc','dv_cc','q_cv']].describe())
cycles.head()


## 3. Corrected SOH formula (voltage-range corrected CC + linear CV)

Reference values are the means of the first 10 segments of `q_cc`, `q_cv`, and `dv_cc`. The CC contribution is corrected by `(dv_avg / dv_cc)^alpha` to make narrow vs wide voltage segments comparable. After computing the raw per-segment SOH, we drop non-finite rows and apply a 5-segment rolling median as the training target.


In [ ]:
# --- reference values from the first 10 segments ---
REF_HEAD = 10
Q_CC_AVG    = cycles["q_cc"].head(REF_HEAD).mean()
Q_CV_AVG    = cycles["q_cv"].head(REF_HEAD).mean()
DELTA_V_AVG = cycles["dv_cc"].head(REF_HEAD).mean()
print(f"Q_cc_avg    = {Q_CC_AVG:.4f}  (Ah)")
print(f"Q_cv_avg    = {Q_CV_AVG:.4f}  (Ah)")
print(f"DeltaV_avg  = {DELTA_V_AVG:.4f}  (V)")

# --- correction + SOH parameters ---
ALPHA = 0.8
W_CC  = 0.6
W_CV  = 1.0 - W_CC   # = 0.4
ROLL_WINDOW = 5

# --- voltage-range correction factor (applied to CC contribution) ---
cycles["K_cc_corr1"] = DELTA_V_AVG / cycles["dv_cc"]
cycles["Q_cc_corr"]  = cycles["K_cc_corr1"].pow(ALPHA)

# --- corrected SOH components and total ---
cycles["soh_cc_corr"]  = cycles["Q_cc_corr"] * (cycles["q_cc"] / Q_CC_AVG)
cycles["soh_cv_corr"]  = cycles["q_cv"] / Q_CV_AVG
cycles["soh_pct_raw"]  = (cycles["soh_cc_corr"] * W_CC + cycles["soh_cv_corr"] * W_CV) * 100.0

# Drop non-finite raw SOH (e.g. dv_cc == 0), re-index
n_before = len(cycles)
cycles   = cycles[np.isfinite(cycles["soh_pct_raw"])].reset_index(drop=True)
cycles["cycle_idx"] = np.arange(len(cycles))
print(f"Dropped segments with non-finite SOH: {n_before - len(cycles)}")
print(f"Segments remaining: {len(cycles)}")

# Rolling-median smoothing -> training/forecasting target
cycles["soh_pct"] = (
    cycles["soh_pct_raw"]
    .rolling(ROLL_WINDOW, min_periods=1, center=True)
    .median()
)
print(cycles[["soh_cc_corr","soh_cv_corr","soh_pct_raw","soh_pct"]].describe())

plt.figure(figsize=(10, 4))
plt.plot(cycles["cycle_idx"], cycles["soh_pct_raw"], color="lightgray", alpha=0.7, label="raw corrected SOH")
plt.plot(cycles["cycle_idx"], cycles["soh_pct"],     color="C1", lw=1.5, label=f"smoothed (window={ROLL_WINDOW})")
plt.xlabel("segment index"); plt.ylabel("SOH %"); plt.legend()
plt.title(f"Corrected SOH (α={ALPHA}, w_cc={W_CC}, w_cv={W_CV}) — NASA Cell #23")
plt.show()


## 4. Build windows and temporal split

In [ ]:
FEATURE_COLS = [
    "duration_min",
    "v_start", "v_end", "delta_v", "v_mean", "v_max", "v_min",
    "i_mean", "i_max", "i_min",
    "temp_mean", "temp_max", "temp_min", "temp_spread",
    "cap_charged_ah", "energy_wh", "n_samples",
    "q_cc", "dv_cc", "q_cv",
    "soh_cc_corr", "soh_cv_corr",
    "K_cc_corr1", "Q_cc_corr",
    "days_since_start",
    "soh_pct",   # current SOH is also an input feature for the next-segment forecast
]
WINDOW = 30
HORIZON = 1
TRAIN_FRAC = 0.75

X_all = cycles[FEATURE_COLS].values.astype(np.float32)
y_all = cycles["soh_pct"].values.astype(np.float32)

split = int(len(cycles) * TRAIN_FRAC)
print(f"Train segments 0..{split-1}  |  Test segments {split}..{len(cycles)-1}")

scaler = StandardScaler().fit(X_all[:split])
X_scaled = scaler.transform(X_all).astype(np.float32)

def make_windows(X, y, window, horizon, start, end):
    Xs, ys = [], []
    for i in range(start, end - window - horizon + 1):
        Xs.append(X[i:i+window])
        ys.append(y[i + window + horizon - 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_windows(X_scaled, y_all, WINDOW, HORIZON, 0, split)
X_test,  y_test  = make_windows(X_scaled, y_all, WINDOW, HORIZON, split - WINDOW, len(cycles))
print("Train windows:", X_train.shape, "Test windows:", X_test.shape)


In [ ]:
class CycleWindowDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

val_n = max(16, int(0.1 * len(X_train)))
train_ds = CycleWindowDS(X_train[:-val_n], y_train[:-val_n])
val_ds   = CycleWindowDS(X_train[-val_n:], y_train[-val_n:])
test_ds  = CycleWindowDS(X_test, y_test)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)
test_dl  = DataLoader(test_ds,  batch_size=64)
print("train", len(train_ds), "val", len(val_ds), "test", len(test_ds))


## 5. Transformer encoder model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class SOHTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, layers=3, dim_ff=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
    def forward(self, x):
        h = self.pos(self.input_proj(x))
        h = self.encoder(h)
        return self.head(h[:, -1]).squeeze(-1)

model = SOHTransformer(n_features=len(FEATURE_COLS)).to(DEVICE)
print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 6. Train

In [ ]:
EPOCHS = 80
LR = 1e-3
PATIENCE = 12

opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.MSELoss()

best_val = float("inf"); best_state = None; bad = 0
hist = {"train": [], "val": []}

for epoch in range(EPOCHS):
    model.train(); tr = 0.0; n = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr += loss.item() * len(xb); n += len(xb)
    sched.step(); tr /= n

    model.eval(); vl = 0.0; nv = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl += loss_fn(model(xb), yb).item() * len(xb); nv += len(xb)
    vl /= nv
    hist["train"].append(tr); hist["val"].append(vl)

    if vl < best_val - 1e-6:
        best_val = vl
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train {tr:.5f}  val {vl:.5f}  best {best_val:.5f}")
    if bad >= PATIENCE:
        print(f"Early stop at epoch {epoch}, best val {best_val:.5f}")
        break

model.load_state_dict(best_state)

plt.figure(figsize=(7, 3))
plt.plot(hist["train"], label="train")
plt.plot(hist["val"],   label="val")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Loss")
plt.show()


## 7. Evaluate

Compare the Transformer against two baselines:
- **Persistence** — predict next SOH = current SOH (last position of the window).
- **Linear extrapolation** — least-squares linear fit on training segments.


In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval()
    out = []
    for xb, _ in dl:
        out.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

y_pred = predict(model, test_dl)

# Persistence: last SOH in each test window (undo standardisation)
soh_idx = FEATURE_COLS.index("soh_pct")
mu, sg = scaler.mean_[soh_idx], scaler.scale_[soh_idx]
y_persist = X_test[:, -1, soh_idx] * sg + mu

# Linear extrapolation fit on training segments
train_idx = np.arange(split)
coef = np.polyfit(train_idx, y_all[:split], 1)
test_cycle_idx = np.arange(split, len(cycles))[-len(y_test):]
y_linear = np.polyval(coef, test_cycle_idx)

def report(name, yt, yp):
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    print(f"{name:14s}  MAE {mean_absolute_error(yt, yp):.4f}  RMSE {rmse:.4f}  R2 {r2_score(yt, yp):.4f}")

report("Transformer", y_test, y_pred)
report("Persistence", y_test, y_persist)
report("Linear",      y_test, y_linear)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cycles["cycle_idx"], cycles["soh_pct"], color="lightgray", label="full SOH series")
plt.plot(test_cycle_idx, y_test,    "k",     lw=2, label="test true")
plt.plot(test_cycle_idx, y_pred,    "C0",    label="Transformer")
plt.plot(test_cycle_idx, y_linear,  "C2--",  label="Linear")
plt.plot(test_cycle_idx, y_persist, "C3:",   label="Persistence")
plt.axvline(split, color="red", ls="--", lw=1, label="train/test split")
plt.xlabel("segment index"); plt.ylabel("SOH %")
plt.title("SOH forecast — NASA Cell #23 (corrected CC/CV, smoothed window=5)")
plt.legend()
plt.show()


## Notes & next steps

- **`ALPHA`** (default 0.8) controls how strongly the voltage-range correction lifts narrow-segment CC contributions. `ALPHA = 0` disables it (no correction); `ALPHA = 1` linear correction; higher values amplify it.
- **`W_CC` / `W_CV`** (defaults 0.6 / 0.4) set how much each phase contributes. Adjust freely; both should sum to 1.
- **`REF_HEAD = 10`** controls how many early segments define the reference. If the first 10 segments aren't representative of "fresh cell" behaviour (e.g. they include any anomalous warm-up cycles), shift to a different slice or pick specific segment IDs.
- **`V_CV`** still controls CC/CV split (4.199 V default). All caveats from the unsmoothed notebook apply.
- **Predicted-Q reconstruction:** if you later want to forecast `q_cc` and `q_cv` directly and rebuild SOH from those forecasts (the `_pred` block in your formula spec), the model head needs to output 2 values instead of 1 and `make_windows` needs to return a 2-D target. Happy to spin a separate notebook for that.
